In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [16]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os
 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cuda
PyTorch version: 2.10.0+cu128


In [17]:
print("Loading DeBERTa model...")
deberta_tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-small", 
    num_labels=5
).to(device).eval()
 
print("Loading RoBERTa model...")
roberta_tok = AutoTokenizer.from_pretrained("roberta-base")
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=5
).to(device).eval()

Loading DeBERTa model...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight     

Loading RoBERTa model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
print(f"Test data shape: {test.shape}")
print(f"Columns: {test.columns.tolist()}")
print("\nFirst few rows:")
print(test.head(2))
 
LABELS = ["A", "B", "C", "D", "E"]
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}

Test data shape: (2000, 8)
Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']

First few rows:
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   C  \
0  Martin Heidegger does not believe in the exist...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   D  \
0  Martin Heidegger believes that the relationshi...   
1  Accelerator-based light-ion fusion is a techni...   

                                      

In [19]:
def build_prompt(row, prefix=""):
    if "prompt" in row.index:
        q = row["prompt"]
    elif "question" in row.index:
        q = row["question"]
    else:
        raise KeyError("Row must have either 'prompt' or 'question' column")
    
    opts = [row[label] for label in LABELS]
    
    text = prefix + q + " "
    for label, option in zip(LABELS, opts):
        text += f" {label}) {option}"
    
    return text
 
 
def get_probs(model, tok, text, device="cuda"):
    enc = tok(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to(device)
    
    with torch.no_grad():
        logits = model(**enc).logits
    
    probs = F.softmax(logits, dim=-1).cpu().numpy()[0]
    return probs
 
 
def get_top1(probs):
    idx = np.argmax(probs)
    return LABELS[idx], probs[idx]
 
 
def get_top3(probs):
    idx = np.argsort(-probs)[:3]
    return [LABELS[i] for i in idx]
 
 
def top3_string(probs):
    return " ".join(get_top3(probs))
 

In [20]:
row25 = test.iloc[25]
print(f"\nRow 25 data:")
print(f"  ID: {row25.get('id', 'N/A')}")
print(f"  Question: {row25.get('question', row25.get('prompt', 'N/A'))[:100]}...")

text25 = build_prompt(row25)
p_deberta_25 = get_probs(deberta_model, deberta_tok, text25, device=device)
 
top1_label, top1_prob = get_top1(p_deberta_25)
 
print(f"\nDeBERTa probabilities: {p_deberta_25}")
print(f"\n✅ ANSWER Q1: {top1_label}, {top1_prob:.6f}")
 
Q1_answer = f"{top1_label}, {top1_prob:.6f}"


Row 25 data:
  ID: 26
  Question: Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully....

DeBERTa probabilities: [0.2617 0.2155 0.2191 0.1637 0.1399]

✅ ANSWER Q1: A, 0.261719


In [21]:
p_roberta_25 = get_probs(roberta_model, roberta_tok, text25, device=device)
print(f"RoBERTa probabilities: {p_roberta_25}")

RoBERTa probabilities: [0.2027849  0.21117158 0.23023415 0.17738865 0.17842074]


In [22]:
avg_probs = (p_deberta_25 + p_roberta_25) / 2
print(f"\nAveraged probabilities: {avg_probs}")
 
top1_avg_label, top1_avg_prob = get_top1(avg_probs)
print(f"\n ANSWER Q2: {top1_avg_label}")
 
Q2_answer = top1_avg_label


Averaged probabilities: [0.23225182 0.21331283 0.22467518 0.17054248 0.15915665]

 ANSWER Q2: A


In [23]:
w_deberta = 0.70
w_roberta = 0.30

weighted_probs_25 = w_deberta * p_deberta_25 + w_roberta * p_roberta_25
print(f"\nWeights: DeBERTa={w_deberta}, RoBERTa={w_roberta}")
print(f"Weighted probabilities: {weighted_probs_25}")
 
top1_weighted_label, top1_weighted_prob = get_top1(weighted_probs_25)
print(f"\n ANSWER Q3: {top1_weighted_label}")
 
Q3_answer = top1_weighted_label


Weights: DeBERTa=0.7, RoBERTa=0.3
Weighted probabilities: [0.244063   0.21423039 0.22251263 0.16784063 0.15148765]

 ANSWER Q3: A


In [24]:

top3_25 = top3_string(weighted_probs_25)
print(f"\nTop-3 predictions: {top3_25}")
print(f"(using weighted ensemble probabilities)")
 
print(f"\n✅ ANSWER Q4: {top3_25}")
 
Q4_answer = top3_25


Top-3 predictions: A C B
(using weighted ensemble probabilities)

✅ ANSWER Q4: A C B


In [25]:
results = []
 
for i, row in test.iterrows():
    if i % 100 == 0:
        print(f"Processing row {i}/{len(test)}...", end='\r')
    
    text = build_prompt(row)
    
    pd_ = get_probs(deberta_model, deberta_tok, text, device=device)
    pr_ = get_probs(roberta_model, roberta_tok, text, device=device)
    
    wp = w_deberta * pd_ + w_roberta * pr_

    pred = top3_string(wp)
    
    results.append({
        "id": row.get("id", i),
        "prediction": pred
    })
 
sub = pd.DataFrame(results)
 
sub.to_csv("submission.csv", index=False)
print(f"\n Submission saved to submission.csv")
print(f" ANSWER Q5: {len(sub)}")
 
Q5_answer = len(sub)
 
print(f"\nFirst 5 rows of submission:")
print(sub.head())
 

Processing row 1900/2000...
 Submission saved to submission.csv
 ANSWER Q5: 2000

First 5 rows of submission:
   id prediction
0   1      A C B
1   2      B C A
2   3      A C B
3   4      B A C
4   5      C A B


In [26]:
tta_diff_count = 0
prefix_tta = "Answer the following multiple-choice question carefully: "
 
for i in range(50):
    row = test.iloc[i]
    
    text_orig = build_prompt(row)
    text_aug = build_prompt(row, prefix=prefix_tta)

    p_orig = get_probs(deberta_model, deberta_tok, text_orig, device=device)
    p_aug = get_probs(deberta_model, deberta_tok, text_aug, device=device)

    p_tta = (p_orig + p_aug) / 2

    top1_orig = np.argmax(p_orig)
    top1_tta = np.argmax(p_tta)
    
    if top1_orig != top1_tta:
        tta_diff_count += 1
    
    if (i + 1) % 10 == 0:
        print(f"Processed {i+1}/50 rows...", end='\r')
 
print(f"\n ANSWER Q6: {tta_diff_count}")


Processed 50/50 rows...
 ANSWER Q6: 16


In [27]:
diff_top1 = 0
positive_gain = 0
diff_top3 = 0
 
for i in range(100):
    row = test.iloc[i]
    text = build_prompt(row)
    
    # Get probabilities
    pd_ = get_probs(deberta_model, deberta_tok, text, device=device)
    pr_ = get_probs(roberta_model, roberta_tok, text, device=device)
    wp = w_deberta * pd_ + w_roberta * pr_
    
    # Q7: Compare top-1
    top1_deberta = np.argmax(pd_)
    top1_ensemble = np.argmax(wp)
    if top1_deberta != top1_ensemble:
        diff_top1 += 1
    
    # Q8: Compare confidence gain
    conf_deberta = np.max(pd_)
    conf_ensemble = np.max(wp)
    gain = conf_ensemble - conf_deberta
    if gain > 0:
        positive_gain += 1
    
    # Q9: Compare top-3
    top3_deberta = top3_string(pd_)
    top3_ensemble = top3_string(wp)
    if top3_deberta != top3_ensemble:
        diff_top3 += 1
    
    if (i + 1) % 20 == 0:
        print(f"Processed {i+1}/100 rows...", end='\r')
 
print(f"\n ANSWER Q7: {diff_top1}")
print(f"   (Top-1 predictions differ between DeBERTa and Ensemble)")
 
print(f"\n ANSWER Q8: {positive_gain}")
print(f"   (Rows with positive confidence gain)")
 
print(f"\n ANSWER Q9: {diff_top3}")
print(f"   (Top-3 predictions differ between DeBERTa and Ensemble)")

 

Processed 100/100 rows...
 ANSWER Q7: 12
   (Top-1 predictions differ between DeBERTa and Ensemble)

 ANSWER Q8: 3
   (Rows with positive confidence gain)

 ANSWER Q9: 25
   (Top-3 predictions differ between DeBERTa and Ensemble)


In [28]:
if "answer" not in test.columns:
    print("⚠️  WARNING: 'answer' column not found in test.csv")
    print(f"Available columns: {test.columns.tolist()}")
    print("\nAssuming 'answer' column for MAP@3 calculation...")
    print("If column name is different, update the code accordingly.\n")
 
def apk(actual, predicted, k=3):
    if actual in predicted[:k]:
        return 1.0 / (predicted.index(actual) + 1)
    return 0.0
 
ap_scores = []
 
for i in range(100):
    row = test.iloc[i]
    text = build_prompt(row)
    
    pd_ = get_probs(deberta_model, deberta_tok, text, device=device)
    pr_ = get_probs(roberta_model, roberta_tok, text, device=device)
    wp = w_deberta * pd_ + w_roberta * pr_

    pred_labels = get_top3(wp)

    actual_label = row.get("answer", None)
    
    if actual_label is None:
        print(f" Row {i}: 'answer' value is missing, skipping...")
        continue
    
    # Calculate AP@3
    ap = apk(actual_label, pred_labels, k=3)
    ap_scores.append(ap)
    
    if (i + 1) % 20 == 0:
        print(f"Processed {i+1}/100 rows...", end='\r')
 
# Calculate mean
if ap_scores:
    map3 = np.mean(ap_scores)
    print(f"\nANSWER Q10: {map3:.4f}")
    print(f"   (MAP@3 calculated from {len(ap_scores)} samples)")
else:
    map3 = None
    print("\nERROR: Could not calculate MAP@3. Check 'answer' column.")
 
Q10_answer = map3

Processed 100/100 rows...
ANSWER Q10: 0.3950
   (MAP@3 calculated from 100 samples)
